# Amortised posteriors

Train once on simulated (observation, function) pairs. After that, a new observation costs you one ODE solve per draw — no retraining, no MCMC, no burn-in, and the draws are independent.
```{note}
Tiny on purpose — a couple dozen modes, a couple thousand steps, a minute or two on a CPU. It's
here to show the API, not to be impressive. Real ones live in `FuncyFlows/examples/`.
```

In [ ]:
import matplotlib.pyplot as plt
import torch

from FuncyFlows.base_measures import CosineBasis, GaussianReferenceMeasure
from FuncyFlows.transports.continuous import (ContinuousTransformation, SumField,
                                              LinearField, MatrixField, DataConditioner)
from FuncyFlows.objectives import ConditionalFlowMatching
from FuncyFlows.utils.train import train

torch.manual_seed(2)
DTYPE = torch.float64
M, NUM_OBS, NOISE = 24, 10, 0.1

basis = CosineBasis(M, dtype=DTYPE)
prior = GaussianReferenceMeasure(basis, alpha=0.05, power=2.0, dtype=DTYPE)

# A FIXED observation operator: amortisation needs one design matrix, not a new one per case.
obs_points = torch.rand(NUM_OBS, 1, dtype=DTYPE)
design_obs = basis.evaluate(obs_points)

## The simulator

`simulate(batch)` returns `(target coefficients, context)`. Context is whatever summary you want the flow to condition on; here it's just the whitened observations, which is plenty because the map is linear.

For anything harder you'd put more thought into this — the context is where all the design work lives in an amortised setup.

In [ ]:
def simulate(batch):
    coeffs = prior.sample(batch)
    observations = coeffs @ design_obs.T + NOISE * torch.randn(batch, NUM_OBS, dtype=DTYPE)
    return coeffs, observations / NOISE          # whiten the context

## A conditional flow

`LinearField(M, context_dim)` picks up a data-dependent drift, and `DataConditioner` lets the context shift the tanh layer's bias. Since none of that depends on `v`, the Jacobian is unchanged and the trace stays exact — you get conditioning for free, as far as the density is concerned.

In [ ]:
field = SumField(
    LinearField(M, NUM_OBS, num_time_modes=4, mode_scale=prior.scale, dtype=DTYPE),
    MatrixField(DataConditioner(M, 96, NUM_OBS, num_time_modes=4, dtype=DTYPE),
                mode_scale=prior.scale),
)
flow = ContinuousTransformation(prior, field, num_steps=12)

objective = ConditionalFlowMatching(flow, simulate, prior.sample, batch_size=128,
                                    weights=1 / prior.scale)
losses = train(objective, flow.parameters(), num_steps=3000, learning_rate=3e-3)
print(f"loss {sum(losses[:50]) / 50:.2f} -> {sum(losses[-50:]) / 50:.2f}")

## One new observation, a thousand draws

In [ ]:
truth = prior.sample(1)
data = truth @ design_obs.T + NOISE * torch.randn(1, NUM_OBS, dtype=DTYPE)
context = data / NOISE

with torch.no_grad():
    draws = flow.transport(prior.sample(1000), context.expand(1000, -1))

precision = torch.diag(1 / prior.variances) + design_obs.T @ design_obs / NOISE ** 2
exact_mean = torch.linalg.solve(precision, design_obs.T @ data[0] / NOISE ** 2)
exact_std = torch.linalg.inv(precision).diagonal().sqrt()

print("mean error", (draws.mean(0) - exact_mean).norm().item())
print("std ratio ", (draws.std(0) / exact_std)[:5].tolist())

In [ ]:
grid = torch.linspace(0, 1, 300, dtype=DTYPE)[:, None]
design = basis.evaluate(grid)
values = draws @ design.T
fig, ax = plt.subplots(figsize=(7, 3))
ax.fill_between(grid[:, 0], values.quantile(0.05, dim=0), values.quantile(0.95, dim=0),
                alpha=0.25, label="flow 5-95%")
ax.plot(grid[:, 0], (truth @ design.T)[0], "k", lw=1.5, label="truth")
ax.plot(grid[:, 0], exact_mean @ design.T, "C3--", lw=1.5, label="exact mean")
ax.scatter(obs_points[:, 0], data[0], color="C3", s=14, zorder=3)
ax.legend(fontsize=8)
plt.show()

That's the trade amortisation makes: one long training run, then near-free inference for every future observation from the same simulator. If you only ever have one observation it's not worth it — notebook `03` is your route. If you've got a thousand, it very much is.

---

Next: [being exact, and checking you're calibrated](05_latent_pcn_and_diagnostics.ipynb).